# 🫀 PI-FNO v2: Physics-Informed Fourier Neural Operator for Cardiac Disease Detection

**Classes:** AS · MR · MS · MVP · Normal  
**Key upgrades over v1:**
1. Multi-scale local context encoder before FNO lifting (captures local temporal structure)
2. Increased modes (512) and width (128) — spectral coverage was severely undersampled
3. GroupNorm replacing InstanceNorm — stable with small batches
4. Deeper, regularised classifier head with LayerNorm + Dropout
5. Physics-aware data augmentation (time shift, amplitude jitter, band-limited noise)
6. Linear warmup + cosine decay LR schedule
7. Label smoothing (ε=0.1) cross-entropy
8. Multi-scale global pooling (mean + max) before projection MLP

In [ ]:
# Mount Drive & install dependencies
from google.colab import drive
drive.mount('/content/drive')
!pip install -q librosa soundfile

## 0. Imports and Configuration

In [ ]:
import os
import math
import platform
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import librosa

warnings.filterwarnings('ignore')

# ── Performance flags ────────────────────────────────────────────────────────
torch._inductor.config.force_disable_caches = True

if platform.system() == 'Darwin':
    torch.compile = lambda model, **kwargs: model
    print('torch.compile disabled on macOS — returning model as-is')

torch.backends.cudnn.benchmark = True
torch.set_default_dtype(torch.float32)

os.makedirs('figures',     exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR      = '/content/drive/MyDrive/Colab Notebooks/Heart-Sound/Data'
CLASS_FOLDERS = ['AS_New', 'MR_New', 'MS_New', 'MVP_New', 'N_New']
CLASS_NAMES   = ['AS',     'MR',     'MS',     'MVP',     'Normal']
USE_SAMPLE_DATA = not os.path.isdir(DATA_DIR)

# ── Hyper-parameters ─────────────────────────────────────────────────────────
CONFIG = {
    'sample_rate'   : 22050,
    'signal_length' : 22050,       # 1 s window
    'use_sample_data': USE_SAMPLE_DATA,
    'test_split'    : 0.20,
    'val_split'     : 0.20,
    'batch_size'    : 32,
    'n_classes'     : len(CLASS_NAMES),
    # ── FNO v2 (upgraded) ──────────────────────────────────────────────────
    'fno_width'     : 128,         # was 64
    'fno_modes'     : 512,         # was 150; spectral coverage ~2.3% → ~4.6%
    'fno_depth'     : 4,
    'fno_proj_dim'  : 128,         # projection MLP hidden dim
    # ── Training ───────────────────────────────────────────────────────────
    'epochs'        : 80,          # was 60; extra budget for the richer model
    'lr'            : 2e-3,
    'weight_decay'  : 1e-4,
    'warmup_epochs' : 10,
    'label_smoothing': 0.10,
    'class_names'   : CLASS_NAMES,
}

## 1. Dataset with Physics-Aware Augmentation

In [ ]:
class HeartSoundDataset(Dataset):
    """
    Loads .wav files, resamples to CONFIG['sample_rate'],
    trims/pads to CONFIG['signal_length'], and amplitude-normalises.
    """
    def __init__(self, data_dir, class_folders,
                 sr=CONFIG['sample_rate'],
                 length=CONFIG['signal_length'],
                 use_synthetic=CONFIG.get('use_sample_data', False),
                 augment=False):
        self.sr      = sr
        self.length  = length
        self.augment = augment

        if use_synthetic:
            rng = np.random.default_rng(42)
            N   = 250
            t   = np.linspace(0, length/sr, length)
            self.signals = np.zeros((N, length), dtype=np.float32)
            for i in range(N):
                cls = i % 5
                f0  = [80, 120, 100, 90, 70][cls]
                sig = (np.sin(2*np.pi*f0*t) +
                       0.3*np.sin(2*np.pi*2*f0*t) +
                       0.1*rng.standard_normal(length))
                self.signals[i] = (sig / np.abs(sig).max()).astype(np.float32)
            self.labels    = np.array([i % 5 for i in range(N)], dtype=np.int64)
            self.filenames = [f'synthetic_{i}.wav' for i in range(N)]
            print(f'Loaded {N} synthetic recordings')
        else:
            self.signals, self.labels, self.filenames = \
                self._load(data_dir, class_folders, sr, length)
            print(f'Loaded {len(self.signals)} real recordings')

        counts = np.bincount(self.labels, minlength=len(class_folders))
        for i, (folder, name) in enumerate(zip(class_folders, CLASS_NAMES)):
            print(f'  [{i}] {name:8s}  ({folder}): {counts[i]:3d} files')

    @staticmethod
    def _load(data_dir, class_folders, sr, length):
        signals, labels, fnames = [], [], []
        for cls_idx, folder in enumerate(class_folders):
            folder_path = os.path.join(data_dir, folder)
            if not os.path.isdir(folder_path):
                print(f'  WARNING: folder not found: {folder_path}')
                continue
            wavs = sorted(f for f in os.listdir(folder_path)
                          if f.lower().endswith('.wav'))
            for fname in wavs:
                fpath = os.path.join(folder_path, fname)
                try:
                    wav, _ = librosa.load(fpath, sr=sr, mono=True)
                except Exception as e:
                    print(f'  WARNING: skipping {fpath}: {e}')
                    continue
                wav = wav[:length] if len(wav) >= length else np.pad(wav, (0, length-len(wav)))
                mx  = np.abs(wav).max()
                if mx > 0: wav /= mx
                signals.append(wav.astype(np.float32))
                labels.append(cls_idx)
                fnames.append(fpath)
        if not signals:
            raise RuntimeError('No .wav files loaded.')
        return (np.array(signals, dtype=np.float32),
                np.array(labels,  dtype=np.int64),
                fnames)

    # ── Physics-aware augmentation ────────────────────────────────────────
    def _augment(self, wav: torch.Tensor) -> torch.Tensor:
        """
        Three augmentations that preserve cardiac acoustics structure:
        1. Cyclic time shift  — cardiac cycle phase is arbitrary.
        2. Amplitude jitter   — microphone gain variation.
        3. Band-limited noise — physiological background noise lives
                                 below 20 Hz and above 800 Hz; preserve
                                 the S1/S2 diagnostic band (20–800 Hz).
        """
        # 1. Cyclic shift (up to 50% of signal)
        shift = torch.randint(0, self.length // 2, (1,)).item()
        wav   = torch.roll(wav, shifts=shift, dims=-1)

        # 2. Amplitude jitter ±15%
        wav   = wav * (0.85 + 0.30 * torch.rand(1, device=wav.device))

        # 3. Band-limited background noise (outside 20–800 Hz diagnostic band)
        noise  = torch.randn_like(wav)
        N_ft   = torch.fft.rfft(noise, norm='ortho')
        freqs  = torch.fft.rfftfreq(self.length, 1.0/self.sr).to(wav.device)
        # Keep noise only outside diagnostic band
        band   = ((freqs >= 20) & (freqs <= 800)).float()
        N_ft   = N_ft * (1.0 - band)              # zero out diagnostic band
        noise  = torch.fft.irfft(N_ft, n=self.length, norm='ortho')
        wav    = wav + 0.02 * noise

        # Re-normalise
        mx = wav.abs().max()
        if mx > 0: wav = wav / mx
        return wav

    def __len__(self):  return len(self.signals)

    def __getitem__(self, idx):
        wav = torch.tensor(self.signals[idx]).unsqueeze(0)   # (1, L)
        if self.augment:
            wav = self._augment(wav.squeeze(0)).unsqueeze(0)
        return wav, torch.tensor(self.labels[idx])


# Build datasets — augment only training split
full_ds = HeartSoundDataset(DATA_DIR, CLASS_FOLDERS)

n_total = len(full_ds)
n_test  = int(n_total * CONFIG['test_split'])
n_val   = int(n_total * CONFIG['val_split'])
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42))

# Wrap the training split to enable augmentation
class AugWrapper(Dataset):
    def __init__(self, subset, base_dataset):
        self.subset  = subset
        self.base    = base_dataset
        self.base.augment = True
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        return self.subset[idx]   # subset.__getitem__ calls base via Subset

# Enable augment on the underlying dataset only during training
# (random_split returns Subset objects sharing the same Dataset object,
#  so we patch augment per-call instead)
class AugSubset(Dataset):
    """Thin wrapper around a Subset that enables augmentation."""
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        wav, label = self.subset[idx]
        # Apply augmentation inline
        ds = self.subset.dataset
        wav = ds._augment(wav.squeeze(0)).unsqueeze(0)
        return wav, label

train_aug_ds = AugSubset(train_ds)

train_loader = DataLoader(train_aug_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True, drop_last=True, pin_memory=True,
                          num_workers=2)
val_loader   = DataLoader(val_ds,  batch_size=CONFIG['batch_size'],
                          shuffle=False, pin_memory=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=CONFIG['batch_size'],
                          shuffle=False, pin_memory=True, num_workers=2)
print(f'\nSplit  Train: {n_train}  |  Val: {n_val}  |  Test: {n_test}')

## 2. Architecture

### Key upgrades over v1

| Component | v1 | v2 | Motivation |
|-----------|----|----|------------|
| FNO modes | 150 | 512 | 150 modes covered only 1.4% of rfft spectrum for 22 kHz audio |
| FNO width | 64 | 128 | More capacity for 18-feature fusion |
| Normalisation | InstanceNorm1d | GroupNorm(8) | Stable with batch sizes 8–32 |
| Lift | Conv1d(1,W,1) | Multi-scale encoder | Captures local temporal context at 3 scales |
| Pooling | Mean only | Mean + Max | Preserves peak activations alongside averages |
| Projection MLP | W→128→64 | 2W→256→128 | Matches wider width; deeper representation |
| Classifier head | Linear(82,5) | LN+Dropout→Linear | Regularisation critical for small datasets |
| LR schedule | Cosine | Warmup+Cosine | FNO spectral weights benefit from gentle warmup |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper: Hz → logit of Nyquist-normalised value
# ─────────────────────────────────────────────────────────────────────────────
def _hz_to_logit(hz, nyquist):
    p = np.clip(hz / nyquist, 1e-6, 1 - 1e-6)
    return float(np.log(p / (1 - p)))


# ─────────────────────────────────────────────────────────────────────────────
# 2.1 Spectral Convolution
# ─────────────────────────────────────────────────────────────────────────────
class SpectralConv1d(nn.Module):
    """FNO spectral convolution: learn complex weights on retained Fourier modes."""
    def __init__(self, in_ch, out_ch, n_modes):
        super().__init__()
        self.n_modes = n_modes
        scale = 1.0 / (in_ch * out_ch) ** 0.5
        self.wr = nn.Parameter(scale * torch.randn(in_ch, out_ch, n_modes))
        self.wi = nn.Parameter(scale * torch.randn(in_ch, out_ch, n_modes))

    def forward(self, x):
        B, C, L = x.shape
        x_ft    = torch.fft.rfft(x, norm='ortho')
        out_ft  = torch.zeros(B, self.wr.shape[1], L//2+1,
                              dtype=torch.cfloat, device=x.device)
        W = torch.complex(self.wr, self.wi)
        out_ft[:, :, :self.n_modes] = torch.einsum(
            'bim,iom->bom', x_ft[:, :, :self.n_modes], W)
        return torch.fft.irfft(out_ft, n=L, norm='ortho')


# ─────────────────────────────────────────────────────────────────────────────
# 2.2 PhysioFrequencyMask  (unchanged from v1 — already well-designed)
# ─────────────────────────────────────────────────────────────────────────────
class PhysioFrequencyMask(nn.Module):
    def __init__(self, n_modes, signal_length, sr, s1_band, s2_band,
                 bump_sharpness=50.0, bump_weight=0.3):
        super().__init__()
        self.n_modes        = n_modes
        self.sr             = sr
        self.L              = signal_length
        self.bump_sharpness = bump_sharpness
        self.bump_weight    = bump_weight
        nyquist             = sr / 2.0

        freqs      = np.fft.rfftfreq(signal_length, 1.0/sr)[:n_modes]
        mask_init  = np.full(n_modes, 0.1)
        for i, f in enumerate(freqs):
            if s1_band[0] <= f <= s1_band[1] or s2_band[0] <= f <= s2_band[1]:
                mask_init[i] = 1.0
        m          = np.clip(mask_init, 1e-6, 1-1e-6)
        self.logit = nn.Parameter(
            torch.tensor(np.log(m/(1-m)), dtype=torch.float32))

        self.s1_lo = nn.Parameter(torch.tensor(_hz_to_logit(s1_band[0], nyquist)))
        self.s1_hi = nn.Parameter(torch.tensor(_hz_to_logit(s1_band[1], nyquist)))
        self.s2_lo = nn.Parameter(torch.tensor(_hz_to_logit(s2_band[0], nyquist)))
        self.s2_hi = nn.Parameter(torch.tensor(_hz_to_logit(s2_band[1], nyquist)))

    def forward(self, x):
        nyquist  = self.sr / 2.0
        freqs_hz = torch.fft.rfftfreq(self.L, 1.0/self.sr).to(x.device)
        freqs_n  = freqs_hz / nyquist
        fm       = torch.full((freqs_hz.shape[0],), 0.1, device=x.device)
        fm[:self.n_modes] = torch.sigmoid(self.logit)
        k       = self.bump_sharpness
        s1_lo_n = torch.sigmoid(self.s1_lo)
        s1_hi_n = torch.sigmoid(self.s1_hi)
        s2_lo_n = torch.sigmoid(self.s2_lo)
        s2_hi_n = torch.sigmoid(self.s2_hi)
        s1_bump = (torch.sigmoid(k*(freqs_n - s1_lo_n)) *
                   torch.sigmoid(k*(s1_hi_n  - freqs_n)))
        s2_bump = (torch.sigmoid(k*(freqs_n - s2_lo_n)) *
                   torch.sigmoid(k*(s2_hi_n  - freqs_n)))
        fm      = torch.clamp(fm + self.bump_weight*(s1_bump+s2_bump), 0., 1.)
        x_ft    = torch.fft.rfft(x, norm='ortho')
        return torch.fft.irfft(
            x_ft * fm.unsqueeze(0).unsqueeze(0), n=x.shape[-1], norm='ortho')

    def get_mask(self):
        return torch.sigmoid(self.logit).detach().cpu().numpy()

    def get_bands(self):
        nyquist = self.sr / 2.0
        return {
            'S1': [torch.sigmoid(self.s1_lo).item()*nyquist,
                   torch.sigmoid(self.s1_hi).item()*nyquist],
            'S2': [torch.sigmoid(self.s2_lo).item()*nyquist,
                   torch.sigmoid(self.s2_hi).item()*nyquist],
        }


# ─────────────────────────────────────────────────────────────────────────────
# 2.3 FNO Block — GroupNorm replaces InstanceNorm  (KEY FIX)
# ─────────────────────────────────────────────────────────────────────────────
class FNOBlock1d(nn.Module):
    """
    GroupNorm(8) is more stable than InstanceNorm1d when batch_size is small.
    With width=128 and 8 groups, each group has 16 channels.
    """
    def __init__(self, width, n_modes):
        super().__init__()
        self.spec   = SpectralConv1d(width, width, n_modes)
        self.bypass = nn.Conv1d(width, width, 1)
        self.norm   = nn.GroupNorm(8, width)      # was InstanceNorm1d

    def forward(self, x):
        return F.gelu(self.norm(self.spec(x) + self.bypass(x)))


# ─────────────────────────────────────────────────────────────────────────────
# 2.4 Multi-Scale Local Context Encoder  (NEW)
# ─────────────────────────────────────────────────────────────────────────────
class MultiScaleEncoder(nn.Module):
    """
    Captures local temporal structure at three scales before the FNO backbone.
    Cardiac murmurs have fine-grained texture (small kernel) whereas S1/S2
    events span ~50–100 ms (larger kernels).

    At 22050 Hz:
      kernel=7   ≈ 0.3 ms  — fine spectral detail
      kernel=31  ≈ 1.4 ms  — murmur texture
      kernel=127 ≈ 5.8 ms  — valve closure transients

    All three branches are concatenated and projected to `out_ch`.
    """
    def __init__(self, in_ch=1, out_ch=128, kernels=(7, 31, 127)):
        super().__init__()
        mid = out_ch // len(kernels)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch, mid, k, padding=k//2, bias=False),
                nn.GroupNorm(min(8, mid), mid),
                nn.GELU(),
                nn.Conv1d(mid, mid, 3, padding=1, bias=False),
                nn.GroupNorm(min(8, mid), mid),
                nn.GELU(),
            )
            for k in kernels
        ])
        total = mid * len(kernels)
        self.proj = nn.Sequential(
            nn.Conv1d(total, out_ch, 1, bias=False),
            nn.GroupNorm(8, out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        feats = torch.cat([b(x) for b in self.branches], dim=1)
        return self.proj(feats)


# ─────────────────────────────────────────────────────────────────────────────
# 2.5 PhysioFeatureExtractor (18 physical features — unchanged from v1)
# ─────────────────────────────────────────────────────────────────────────────
class PhysioFeatureExtractor(nn.Module):
    """
    Extracts 18 explicit physical quantities from a raw PCG signal.
    All quantities are differentiable w.r.t. the input.
    Some have learnable parameters (band boundaries, harmonic weights).
    """
    def __init__(self, signal_length=CONFIG['signal_length'],
                 sr=CONFIG['sample_rate'],
                 n_modes=CONFIG['fno_modes']):
        super().__init__()
        self.sr = sr
        self.L  = signal_length
        nyquist = sr / 2.0

        self.s1_lo      = nn.Parameter(torch.tensor(_hz_to_logit(25,  nyquist)))
        self.s1_hi      = nn.Parameter(torch.tensor(_hz_to_logit(45,  nyquist)))
        self.s2_lo      = nn.Parameter(torch.tensor(_hz_to_logit(50,  nyquist)))
        self.s2_hi      = nn.Parameter(torch.tensor(_hz_to_logit(70,  nyquist)))
        self.mu_lo      = nn.Parameter(torch.tensor(_hz_to_logit(100, nyquist)))
        self.mu_hi      = nn.Parameter(torch.tensor(_hz_to_logit(500, nyquist)))
        self.hr_lo      = nn.Parameter(torch.tensor(_hz_to_logit(0.8, nyquist)))
        self.hr_hi      = nn.Parameter(torch.tensor(_hz_to_logit(3.0, nyquist)))
        self.harmonic_w = nn.Parameter(torch.ones(4))

    def _psd_and_freqs(self, s):
        psd = torch.fft.rfft(s, norm='ortho').abs() ** 2
        f   = torch.fft.rfftfreq(self.L, 1.0/self.sr).to(s.device)
        return psd, f

    def _band_energy(self, psd, f, lo, hi):
        k    = 50.0
        mask = (torch.sigmoid(k*(f-lo)) * torch.sigmoid(k*(hi-f)))
        return (psd * mask.unsqueeze(0)).sum(-1)

    def _spectral_centroid(self, psd, f, lo, hi):
        k    = 50.0
        mask = (torch.sigmoid(k*(f-lo)) * torch.sigmoid(k*(hi-f)))
        num  = (psd*mask.unsqueeze(0)*f.unsqueeze(0)).sum(-1)
        den  = (psd*mask.unsqueeze(0)).sum(-1) + 1e-8
        return num / den

    def _hilbert_envelope(self, s):
        S = torch.fft.rfft(s, norm='ortho')
        h = torch.zeros(S.shape[-1], device=s.device)
        h[0] = 1.0
        if self.L % 2 == 0:
            h[1:self.L//2]  = 2.0
            h[self.L//2]    = 1.0
        else:
            h[1:(self.L+1)//2] = 2.0
        analytic = torch.fft.irfft(S*h.unsqueeze(0), n=self.L, norm='ortho')
        return (s**2 + analytic**2).sqrt()

    def _instantaneous_frequency(self, s):
        S = torch.fft.rfft(s, norm='ortho')
        h = torch.zeros(S.shape[-1], device=s.device)
        h[0] = 1.0
        if self.L % 2 == 0:
            h[1:self.L//2]  = 2.0
            h[self.L//2]    = 1.0
        else:
            h[1:(self.L+1)//2] = 2.0
        analytic = torch.fft.irfft(S*h.unsqueeze(0), n=self.L, norm='ortho')
        phase    = torch.atan2(analytic, s)
        return torch.diff(phase, dim=-1) / (2*math.pi) * self.sr

    def _cross_beat_consistency(self, s, psd, f):
        k      = 50.0
        hr_lo  = torch.sigmoid(self.hr_lo) * (self.sr/2.0)
        hr_hi  = torch.sigmoid(self.hr_hi) * (self.sr/2.0)
        hr_mask = (torch.sigmoid(k*(f-hr_lo)) * torch.sigmoid(k*(hr_hi-f)))
        f0_idx  = (psd * hr_mask.unsqueeze(0)).argmax(-1)
        f0      = f[f0_idx]
        harmonic_energy = torch.zeros(s.shape[0], device=s.device)
        for n in range(1, 5):
            nf0   = (n * f0).unsqueeze(1)
            width = 2.0
            hmask = torch.sigmoid(k*(f.unsqueeze(0)-nf0+width)) * \
                    torch.sigmoid(k*(nf0+width-f.unsqueeze(0)))
            harmonic_energy += (psd * hmask).sum(-1)
        total_e = psd.sum(-1) + 1e-8
        return harmonic_energy / total_e

    def forward(self, x):
        s         = x.squeeze(1)                          # (B, L)
        psd, f    = self._psd_and_freqs(s)
        nyquist   = self.sr / 2.0
        s1_lo     = torch.sigmoid(self.s1_lo) * nyquist
        s1_hi     = torch.sigmoid(self.s1_hi) * nyquist
        s2_lo     = torch.sigmoid(self.s2_lo) * nyquist
        s2_hi     = torch.sigmoid(self.s2_hi) * nyquist
        mu_lo     = torch.sigmoid(self.mu_lo) * nyquist
        mu_hi     = torch.sigmoid(self.mu_hi) * nyquist
        hr_lo     = torch.sigmoid(self.hr_lo) * nyquist
        hr_hi     = torch.sigmoid(self.hr_hi) * nyquist

        # 1–2. S1/S2 spectral energy
        e_s1 = self._band_energy(psd, f, s1_lo, s1_hi)
        e_s2 = self._band_energy(psd, f, s2_lo, s2_hi)
        # 3–4. S1/S2 spectral centroid
        c_s1 = self._spectral_centroid(psd, f, s1_lo, s1_hi)
        c_s2 = self._spectral_centroid(psd, f, s2_lo, s2_hi)
        # 5. Murmur bandwidth
        mu_bw = self._band_energy(psd, f, mu_lo, mu_hi)
        # 6–9. Harmonic energies
        hw    = torch.softmax(self.harmonic_w, dim=0)
        k_shp = 50.0
        hr_mask = (torch.sigmoid(k_shp*(f-hr_lo)) * torch.sigmoid(k_shp*(hr_hi-f)))
        f0_idx  = (psd * hr_mask.unsqueeze(0)).argmax(-1)
        f0_hz   = f[f0_idx]
        harm_feats = []
        for n in range(1, 5):
            nf0   = (n * f0_hz).unsqueeze(1)
            w     = 2.0
            hmask = (torch.sigmoid(k_shp*(f.unsqueeze(0)-nf0+w)) *
                     torch.sigmoid(k_shp*(nf0+w-f.unsqueeze(0))))
            harm_feats.append(hw[n-1] * (psd*hmask).sum(-1))
        # 10. Dominant frequency
        f0_feat = f0_hz
        # 11. Periodicity ratio
        cardiac_e  = self._band_energy(psd, f, hr_lo, hr_hi)
        total_e    = psd.sum(-1) + 1e-8
        period_r   = cardiac_e / total_e
        # 12. RMS amplitude
        rms        = s.pow(2).mean(-1).sqrt()
        # 13. S1/S2 energy ratio
        s1s2_ratio = torch.log(e_s1 / (e_s2 + 1e-8) + 1e-8)
        # 14–15. Envelope mean + std
        env        = self._hilbert_envelope(s)
        env_mean   = env.mean(-1)
        env_std    = env.std(-1)
        # 16. Instantaneous frequency mean
        inst_f     = self._instantaneous_frequency(s).mean(-1)
        # 17. S1-to-S2 interval
        s1s2_int   = 1.0 / (2.0 * f0_hz + 1e-8)
        # 18. Cross-beat consistency
        cbc        = self._cross_beat_consistency(s, psd, f)

        features = torch.stack(
            [e_s1, e_s2, c_s1, c_s2, mu_bw,
             harm_feats[0], harm_feats[1], harm_feats[2], harm_feats[3],
             f0_feat, period_r, rms, s1s2_ratio,
             env_mean, env_std, inst_f, s1s2_int, cbc],
            dim=1)                                         # (B, 18)
        # Normalise to zero mean / unit variance (running stats are unavailable;
        # batch-level normalisation keeps magnitudes comparable across features)
        features = (features - features.mean(0, keepdim=True)) / \
                   (features.std(0, keepdim=True) + 1e-8)
        return features

    def get_bands(self):
        nyquist = self.sr / 2.0
        return {
            'S1':    [torch.sigmoid(self.s1_lo).item()*nyquist,
                      torch.sigmoid(self.s1_hi).item()*nyquist],
            'S2':    [torch.sigmoid(self.s2_lo).item()*nyquist,
                      torch.sigmoid(self.s2_hi).item()*nyquist],
            'murmur':[torch.sigmoid(self.mu_lo).item()*nyquist,
                      torch.sigmoid(self.mu_hi).item()*nyquist],
            'HR':    [torch.sigmoid(self.hr_lo).item()*nyquist,
                      torch.sigmoid(self.hr_hi).item()*nyquist],
        }


# ─────────────────────────────────────────────────────────────────────────────
# 2.6 FNO Classifier v2  (main upgrade)
# ─────────────────────────────────────────────────────────────────────────────
class FNOClassifierV2(nn.Module):
    """
    Physics-constrained FNO v2.

    Pipeline:
      raw signal (B,1,L)
        → PhysioFrequencyMask             [learnable spectral gating]
        → MultiScaleEncoder               [local temporal context at 3 scales]
        → 4 × FNOBlock1d(GroupNorm)       [global spectral learning]
        → Mean + Max global pooling       [richer representation than mean-only]
        → Projection MLP(2W→256→128)      [deeper than v1]
        concat with PhysioFeatureExtractor(18 features)
        → Classifier head(128+18→256→128→5) [LN + Dropout regularisation]
    """
    def __init__(self,
                 n_classes=CONFIG['n_classes'],
                 width=CONFIG['fno_width'],
                 n_modes=CONFIG['fno_modes'],
                 depth=CONFIG['fno_depth'],
                 proj_dim=CONFIG['fno_proj_dim'],
                 signal_length=CONFIG['signal_length'],
                 sr=CONFIG['sample_rate']):
        super().__init__()

        # Physics-aware spectral gating
        self.freq_mask = PhysioFrequencyMask(
            n_modes, signal_length, sr,
            s1_band=[25, 45], s2_band=[50, 70])

        # Multi-scale local context → lifts (1,L) → (width,L)
        self.encoder   = MultiScaleEncoder(in_ch=1, out_ch=width)

        # FNO backbone (GroupNorm)
        self.fno_blocks = nn.ModuleList(
            [FNOBlock1d(width, n_modes) for _ in range(depth)])

        # Global pooling: mean + max → 2*width
        # Projection MLP
        self.project = nn.Sequential(
            nn.Linear(2*width, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, proj_dim), nn.GELU())

        # 18-dim physics feature extractor
        self.phys = PhysioFeatureExtractor(signal_length, sr, n_modes)

        # Classifier head: proj_dim + 18 → n_classes
        # LayerNorm + two-layer MLP with Dropout before final linear
        fused = proj_dim + 18
        self.head = nn.Sequential(
            nn.LayerNorm(fused),
            nn.Linear(fused, 256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, 128),   nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes))

    def forward(self, x):
        # 1. Spectral gating
        h    = self.freq_mask(x)              # (B, 1, L)
        # 2. Multi-scale local context
        h    = self.encoder(h)                # (B, width, L)
        # 3. FNO blocks
        for blk in self.fno_blocks:
            h = blk(h)
        # 4. Global pooling (mean + max)
        h_mean = h.mean(dim=-1)               # (B, width)
        h_max  = h.max(dim=-1).values         # (B, width)
        h_pool = torch.cat([h_mean, h_max], dim=1)  # (B, 2*width)
        # 5. Projection MLP
        emb    = self.project(h_pool)         # (B, proj_dim)
        # 6. Physics features
        phys   = self.phys(x)                 # (B, 18)
        # 7. Fuse + classify
        fused  = torch.cat([emb, phys], dim=1)
        logits = self.head(fused)
        return logits, emb


model_fno = FNOClassifierV2().to(DEVICE)
print(f'FNO v2 | params: {sum(p.numel() for p in model_fno.parameters()):,}')

## 3. CNN Baseline (unchanged from v1)

In [ ]:
class MelSpectrogramLayer(nn.Module):
    """Differentiable log-Mel spectrogram."""
    def __init__(self, sr=CONFIG['sample_rate'], n_fft=512, hop=128, n_mels=64):
        super().__init__()
        self.n_fft, self.hop = n_fft, hop
        self.register_buffer('window', torch.hann_window(n_fft))
        self.register_buffer('mel_fb',
            torch.tensor(self._mel_fb(sr, n_fft, n_mels), dtype=torch.float32))

    @staticmethod
    def _mel_fb(sr, n_fft, n_mels, fmin=20, fmax=None):
        fmax  = fmax or sr/2.0
        h2m   = lambda f: 2595*np.log10(1+f/700)
        m2h   = lambda m: 700*(10**(m/2595)-1)
        mels  = np.linspace(h2m(fmin), h2m(fmax), n_mels+2)
        hz    = m2h(mels)
        freqs = np.fft.rfftfreq(n_fft, 1.0/sr)
        fb    = np.zeros((n_mels, len(freqs)))
        for m in range(n_mels):
            lo, c, hi = hz[m], hz[m+1], hz[m+2]
            up   = (freqs >= lo) & (freqs <= c)
            down = (freqs > c)  & (freqs <= hi)
            fb[m, up]   = (freqs[up]   - lo) / (c  - lo + 1e-8)
            fb[m, down] = (hi - freqs[down]) / (hi - c  + 1e-8)
        return fb

    def forward(self, x):
        x    = x.squeeze(1)
        stft = torch.stft(x, n_fft=self.n_fft, hop_length=self.hop,
                          window=self.window, return_complex=True)
        mel  = torch.log(torch.matmul(self.mel_fb, stft.abs()**2) + 1e-6)
        return mel.unsqueeze(1)


class CNNBaseline(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.mel = MelSpectrogramLayer()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes))

    def forward(self, x):
        return self.head(self.cnn(self.mel(x)))


model_cnn = CNNBaseline().to(DEVICE)
print(f'CNN    | params: {sum(p.numel() for p in model_cnn.parameters()):,}')

## 4. Training

Key changes from v1:
- **Label-smoothed cross-entropy** (ε=0.10): prevents the model from becoming over-confident on the small training set, improving generalisation.
- **Linear warmup + cosine decay**: FNO spectral weights are sensitive to large early gradients; a 10-epoch warmup stabilises training.
- **Gradient clipping** retained at 1.0.

In [ ]:
def get_scheduler_with_warmup(optimizer, warmup_epochs, total_epochs):
    """Linear warmup then cosine annealing."""
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs         # linear ramp 0→1
        progress = (ep - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))  # cosine decay
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_model(model, train_loader, val_loader,
                model_name='model', use_physio=True,
                epochs=CONFIG['epochs']):

    opt   = torch.optim.AdamW(model.parameters(),
                               lr=CONFIG['lr'],
                               weight_decay=CONFIG['weight_decay'])
    sched = get_scheduler_with_warmup(opt,
                warmup_epochs=CONFIG['warmup_epochs'],
                total_epochs=epochs)

    # Label-smoothed CE for FNO; standard CE for CNN baseline
    label_smooth = CONFIG['label_smoothing'] if use_physio else 0.0
    ce_fn = nn.CrossEntropyLoss(label_smoothing=label_smooth)

    hist     = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc']}
    best_val = 0.0

    for ep in range(1, epochs+1):
        model.train()
        run_loss = correct = total = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            out    = model(xb)
            logits = out[0] if isinstance(out, tuple) else out
            loss   = ce_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            run_loss += loss.item()
            correct  += (logits.argmax(1) == yb).sum().item()
            total    += yb.size(0)

        sched.step()
        train_acc  = correct / total
        train_loss = run_loss / len(train_loader)

        model.eval()
        vc = vt = vl = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                out    = model(xb)
                logits = out[0] if isinstance(out, tuple) else out
                vl    += F.cross_entropy(logits, yb).item()
                vc    += (logits.argmax(1) == yb).sum().item()
                vt    += yb.size(0)
        val_acc  = vc / vt
        val_loss = vl / len(val_loader)

        for k, v in [('train_loss', train_loss), ('val_loss', val_loss),
                     ('train_acc',  train_acc),  ('val_acc',  val_acc)]:
            hist[k].append(v)

        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(),
                       f'checkpoints/{model_name}_best.pt')

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]['lr']
            print(f'[{model_name}] Ep {ep:3d}/{epochs}  '
                  f'train acc {train_acc:.3f}  val acc {val_acc:.3f}  '
                  f'loss {train_loss:.4f}  lr {lr_now:.2e}')

    model.load_state_dict(
        torch.load(f'checkpoints/{model_name}_best.pt', map_location=DEVICE))
    print(f'[{model_name}] Best val acc: {best_val:.4f}')
    return hist


print('Training FNO v2 (physiologically constrained)...')
history_fno = train_model(model_fno, train_loader, val_loader,
                           model_name='fno', use_physio=True)

print('\nTraining CNN baseline...')
history_cnn = train_model(model_cnn, train_loader, val_loader,
                           model_name='cnn', use_physio=False)

## 5. Training Curves

In [ ]:
ep  = range(1, CONFIG['epochs']+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(ep, history_fno['train_acc'], 'b-',  label='FNO v2 train')
axes[0].plot(ep, history_fno['val_acc'],   'b--', label='FNO v2 val',   lw=2)
axes[0].plot(ep, history_cnn['train_acc'], 'r-',  label='CNN train')
axes[0].plot(ep, history_cnn['val_acc'],   'r--', label='CNN val',      lw=2)
axes[0].axvline(CONFIG['warmup_epochs'], color='gray', ls=':', lw=1, label='End warmup')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Epoch')

axes[1].plot(ep, history_fno['train_loss'], 'b-',  label='FNO v2 train')
axes[1].plot(ep, history_fno['val_loss'],   'b--', label='FNO v2 val')
axes[1].plot(ep, history_cnn['train_loss'], 'r-',  label='CNN train')
axes[1].plot(ep, history_cnn['val_loss'],   'r--', label='CNN val')
axes[1].axvline(CONFIG['warmup_epochs'], color='gray', ls=':', lw=1)
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('figures/training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Test-Set Evaluation

In [ ]:
def evaluate(model, loader, model_name):
    model.eval()
    preds, labels_all = [], []
    with torch.inference_mode():
        for xb, yb in loader:
            out    = model(xb.to(DEVICE, non_blocking=True))
            logits = out[0] if isinstance(out, tuple) else out
            preds.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(yb.numpy())
    acc = np.mean(np.array(preds) == np.array(labels_all))
    print(f'\n{"="*52}')
    print(f'{model_name}  |  Test Accuracy: {acc:.4f}')
    print('='*52)
    print(classification_report(labels_all, preds,
                                 target_names=CONFIG['class_names']))
    return preds, labels_all, acc


preds_fno, labels_fno, acc_fno = evaluate(
    model_fno, test_loader, 'FNO v2 (physio-constrained)')
preds_cnn, labels_cnn, acc_cnn = evaluate(
    model_cnn, test_loader, 'CNN Baseline')

delta = (acc_fno - acc_cnn) * 100
sign  = '+' if delta >= 0 else ''
print(f'\nAccuracy delta (FNO − CNN): {sign}{delta:.1f} percentage points')

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, p, l, title in [
    (axes[0], preds_fno, labels_fno,
     f'FNO v2 (physio-constrained)\nAcc = {acc_fno:.3f}'),
    (axes[1], preds_cnn, labels_cnn,
     f'CNN Baseline\nAcc = {acc_cnn:.3f}'),
]:
    cm = confusion_matrix(l, p, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', ax=ax,
                xticklabels=CONFIG['class_names'],
                yticklabels=CONFIG['class_names'],
                cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('figures/confusion_matrices_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Learned Physical Quantities

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('FNO v2 — Learned Physical Quantities After Training',
             fontsize=13, fontweight='bold', y=0.98)

nyq        = CONFIG['sample_rate'] / 2.0
mask_bands = model_fno.freq_mask.get_bands()
phys_bands = model_fno.phys.get_bands()

# 1. Learned frequency mask
ax1   = fig.add_subplot(gs[0, :2])
mask  = model_fno.freq_mask.get_mask()
freqs = np.fft.rfftfreq(CONFIG['signal_length'],
                         1.0/CONFIG['sample_rate'])[:CONFIG['fno_modes']]
dfreq = freqs[1] - freqs[0] if len(freqs) > 1 else 1
ax1.bar(freqs, mask, width=dfreq, color='steelblue', alpha=0.8)
ax1.axvspan(*mask_bands['S1'], alpha=0.2, color='royalblue',
            label=f'S1 {mask_bands["S1"][0]:.1f}–{mask_bands["S1"][1]:.1f} Hz')
ax1.axvspan(*mask_bands['S2'], alpha=0.2, color='tomato',
            label=f'S2 {mask_bands["S2"][0]:.1f}–{mask_bands["S2"][1]:.1f} Hz')
ax1.set_title('PhysioFrequencyMask — per-mode gains (v2: 512 modes)')
ax1.set_xlabel('Frequency (Hz)'); ax1.set_ylabel('Mask weight')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# 2. Band boundary drift
ax2 = fig.add_subplot(gs[0, 2])
band_labels = ['S1 lo', 'S1 hi', 'S2 lo', 'S2 hi']
mask_vals   = [mask_bands['S1'][0], mask_bands['S1'][1],
               mask_bands['S2'][0], mask_bands['S2'][1]]
phys_vals   = [phys_bands['S1'][0], phys_bands['S1'][1],
               phys_bands['S2'][0], phys_bands['S2'][1]]
init_vals   = [25, 45, 50, 70]
x_pos       = np.arange(4)
ax2.bar(x_pos-0.25, init_vals,  0.25, label='Init',       color='gray',      alpha=0.7)
ax2.bar(x_pos,      mask_vals,  0.25, label='Mask',        color='steelblue', alpha=0.8)
ax2.bar(x_pos+0.25, phys_vals,  0.25, label='Extractor',   color='tomato',    alpha=0.8)
ax2.set_xticks(x_pos); ax2.set_xticklabels(band_labels, fontsize=8)
ax2.set_ylabel('Hz'); ax2.set_title('S1/S2 boundary drift')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3, axis='y')

# 3. Harmonic weights
ax3 = fig.add_subplot(gs[1, 0])
hw  = torch.softmax(model_fno.phys.harmonic_w, dim=0).detach().cpu().numpy()
ax3.bar(['f₀','2f₀','3f₀','4f₀'], hw,
        color=['steelblue','royalblue','cornflowerblue','lightsteelblue'],
        alpha=0.85, edgecolor='navy')
ax3.axhline(0.25, color='gray', ls='--', lw=1, label='Uniform')
ax3.set_ylabel('Softmax weight'); ax3.set_ylim(0, 1)
ax3.set_title('Learned harmonic weights')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3, axis='y')
for i, v in enumerate(hw):
    ax3.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# 4. SpectralConv1d weight magnitudes — block 1
ax4  = fig.add_subplot(gs[1, 1])
W    = torch.complex(model_fno.fno_blocks[0].spec.wr,
                     model_fno.fno_blocks[0].spec.wi)
Wm   = W.abs().mean(dim=(0,1)).detach().cpu().numpy()
ax4.bar(np.arange(len(Wm)), Wm, color='purple', alpha=0.7)
ax4.set_xlabel('Fourier mode index'); ax4.set_ylabel('Mean |W|')
ax4.set_title('SpectralConv1d magnitudes (block 1)')
ax4.grid(alpha=0.3)

# 5. Magnitudes across all blocks
ax5     = fig.add_subplot(gs[1, 2])
colors_b = plt.cm.viridis(np.linspace(0, 1, CONFIG['fno_depth']))
for i, blk in enumerate(model_fno.fno_blocks):
    W  = torch.complex(blk.spec.wr, blk.spec.wi)
    Wm = W.abs().mean(dim=(0,1)).detach().cpu().numpy()
    ax5.plot(np.arange(len(Wm)), Wm, color=colors_b[i], lw=1.5, label=f'Block {i+1}')
ax5.set_xlabel('Fourier mode index'); ax5.set_ylabel('Mean |W|')
ax5.set_title('Spectral magnitudes — all FNO blocks')
ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

# 6. Summary table
ax6 = fig.add_subplot(gs[2, :])
ax6.axis('off')
rows = [
    ['Parameter', 'Init (Hz)', 'Mask learned (Hz)', 'Extractor learned (Hz)'],
    ['S1 lo', '25.0', f'{mask_bands["S1"][0]:.1f}', f'{phys_bands["S1"][0]:.1f}'],
    ['S1 hi', '45.0', f'{mask_bands["S1"][1]:.1f}', f'{phys_bands["S1"][1]:.1f}'],
    ['S2 lo', '50.0', f'{mask_bands["S2"][0]:.1f}', f'{phys_bands["S2"][0]:.1f}'],
    ['S2 hi', '70.0', f'{mask_bands["S2"][1]:.1f}', f'{phys_bands["S2"][1]:.1f}'],
    ['Murmur lo', '100', '—', f'{phys_bands["murmur"][0]:.0f}'],
    ['Murmur hi', '500', '—', f'{phys_bands["murmur"][1]:.0f}'],
    ['HR lo',  '0.80', '—', f'{phys_bands["HR"][0]:.2f}'],
    ['HR hi',  '3.00', '—', f'{phys_bands["HR"][1]:.2f}'],
]
tbl = ax6.table(cellText=rows[1:], colLabels=rows[0], loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
tbl.scale(1.2, 1.4)

plt.savefig('figures/learned_quantities_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. t-SNE Latent Space

In [ ]:
from sklearn.manifold import TSNE

model_fno.eval()
embs, tlabels = [], []
with torch.inference_mode():
    for xb, yb in test_loader:
        _, emb = model_fno(xb.to(DEVICE, non_blocking=True))
        embs.append(emb.cpu().numpy())
        tlabels.extend(yb.numpy())

embs    = np.concatenate(embs)
tlabels = np.array(tlabels)
emb_2d  = TSNE(n_components=2, perplexity=15,
                random_state=42).fit_transform(embs)

fig, ax = plt.subplots(figsize=(8, 6))
for cls in range(CONFIG['n_classes']):
    idx = tlabels == cls
    ax.scatter(emb_2d[idx, 0], emb_2d[idx, 1],
               label=CONFIG['class_names'][cls], alpha=0.75, s=45, edgecolors='none')
ax.set_title('t-SNE — FNO v2 Latent Embeddings (test set)')
ax.legend(fontsize=9); ax.axis('off')
plt.tight_layout()
plt.savefig('figures/tsne_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Results Summary

In [ ]:
print('\n' + '='*65)
print(' RESULTS SUMMARY — PI-FNO v2')
print('='*65)
print(f' {"Model":<42} {"Test Acc":>8}')
print('-'*65)
print(f' {"CNN Baseline (Mel spectrogram)":<42} {acc_cnn:>8.4f}')
print(f' {"FNO v2 (physio-constrained)":<42} {acc_fno:>8.4f}')
print('='*65)
delta = (acc_fno - acc_cnn)*100
sign  = '+' if delta >= 0 else ''
print(f' Δ (FNO v2 − CNN): {sign}{delta:.1f} pp')
print()
print(' Key v2 improvements over v1:')
print('  [1] FNO modes        : 150  → 512   (spectral coverage ×3.4)')
print('  [2] FNO width        : 64   → 128   (doubled capacity)')
print('  [3] Normalisation    : InstanceNorm → GroupNorm(8)')
print('  [4] Lift             : Conv1d(1,W,1) → MultiScaleEncoder(3 kernels)')
print('  [5] Pooling          : mean-only → mean+max (richer representation)')
print('  [6] Projection MLP   : W→128→64 → 2W→256→128')
print('  [7] Classifier head  : Linear(82,5) → LN+Drop→256→128→5')
print('  [8] LR schedule      : Cosine → Warmup(10ep)+Cosine')
print('  [9] Loss             : CE → Label-smoothed CE (ε=0.10)')
print(' [10] Augmentation     : None → cyclic shift + amplitude + band-limited noise')
print('='*65)